In [1]:
import pandas as pd
import networkx as nx
import requests
import os
from google.colab import files

In [2]:
# Download PrimeKG dataset (kg.csv) from the official Harvard Dataverse repository
# We use a direct download approach to bypass the Dataverse UI
print("Downloading PrimeKG dataset... (This may take a minute or two)")
url = "https://dataverse.harvard.edu/api/access/datafile/6180620"
filename = "primekg.csv"

if not os.path.exists(filename):
    response = requests.get(url, stream=True)
    with open(filename, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print("Download complete!")
else:
    print("Dataset already exists locally.")

Download complete!


In [3]:
# Load into Pandas DataFrame
df_kg = pd.read_csv(filename, low_memory=False)

# Display the structure of the dataset
print(f"Total edges in PrimeKG: {len(df_kg)}")
df_kg.head()

Total edges in PrimeKG: 8100498


,relation,display_relation,x_index,x_id,x_type,x_name,x_source,y_index,y_id,y_type,y_name,y_source
0,protein_protein,ppi,0,9796,gene/protein,PHYHIP,NCBI,8889,56992,gene/protein,KIF15,NCBI
1,protein_protein,ppi,1,7918,gene/protein,GPANK1,NCBI,2798,9240,gene/protein,PNMA1,NCBI
2,protein_protein,ppi,2,8233,gene/protein,ZRSR2,NCBI,5646,23548,gene/protein,TTC33,NCBI
3,protein_protein,ppi,3,4899,gene/protein,NRF1,NCBI,11592,11253,gene/protein,MAN1B1,NCBI
4,protein_protein,ppi,4,5297,gene/protein,PI4KA,NCBI,2122,8601,gene/protein,RGS20,NCBI


In [4]:
# Updated filter to use 'effect/phenotype'
clinical_df = df_kg[
    ((df_kg['x_type'] == 'disease') & (df_kg['y_type'] == 'effect/phenotype')) |
    ((df_kg['x_type'] == 'effect/phenotype') & (df_kg['y_type'] == 'disease'))
].copy()

print(f"Total clinical edges (Disease-Symptom): {len(clinical_df)}")

# Clean up to keep only what we need for the graph
edges_df = clinical_df[['x_name', 'x_type', 'y_name', 'y_type', 'display_relation']].dropna()

# Normalize text (lowercase) to make entity linking easier
edges_df['x_name'] = edges_df['x_name'].str.lower()
edges_df['y_name'] = edges_df['y_name'].str.lower()

edges_df.head()

Total clinical edges (Disease-Symptom): 303020


,x_name,x_type,y_name,y_type,display_relation
3084053,osteogenesis imperfecta,disease,hearing impairment,effect/phenotype,phenotype absent
3084054,osteogenesis imperfecta,disease,intellectual disability,effect/phenotype,phenotype absent
3084055,autosomal recessive nonsyndromic deafness,disease,vestibular dysfunction,effect/phenotype,phenotype absent
3084056,autosomal recessive nonsyndromic deafness,disease,abnormal facial shape,effect/phenotype,phenotype absent
3084057,autosomal recessive nonsyndromic deafness,disease,visual impairment,effect/phenotype,phenotype absent


In [5]:
# Initialize an undirected graph
G = nx.Graph()

# Add edges directly from the pandas dataframe
G = nx.from_pandas_edgelist(
    edges_df,
    source='x_name',
    target='y_name',
    edge_attr=['display_relation']
)

# Attach node attributes (disease vs effect/phenotype)
node_types = {}
for _, row in edges_df.iterrows():
    node_types[row['x_name']] = row['x_type']
    node_types[row['y_name']] = row['y_type']

nx.set_node_attributes(G, node_types, 'node_type')

print(f"Graph constructed successfully!")
print(f"Total Nodes: {G.number_of_nodes()}")
print(f"Total Edges: {G.number_of_edges()}")

Graph constructed successfully!
Total Nodes: 15999
Total Edges: 151331


In [6]:
def get_next_symptom_candidates(graph, patient_symptoms, top_k_diseases=3):
    possible_diseases = {}

    # Step 1: Find all diseases connected to the patient's current symptoms
    for symptom in patient_symptoms:
        if symptom in graph:
            neighbors = graph.neighbors(symptom)
            for neighbor in neighbors:
                if graph.nodes[neighbor].get('node_type') == 'disease':
                    possible_diseases[neighbor] = possible_diseases.get(neighbor, 0) + 1

    if not possible_diseases:
        return "No matching diseases found in the graph for these symptoms."

    # Step 2: Rank diseases by symptom match count
    sorted_diseases = sorted(possible_diseases.items(), key=lambda x: x[1], reverse=True)
    top_diseases = [d[0] for d in sorted_diseases[:top_k_diseases]]

    # Step 3: Find missing symptoms for the top diseases
    symptoms_to_ask = set()
    for disease in top_diseases:
        for neighbor in graph.neighbors(disease):
            # UPDATED: checking for 'effect/phenotype'
            if graph.nodes[neighbor].get('node_type') == 'effect/phenotype' and neighbor not in patient_symptoms:
                symptoms_to_ask.add(neighbor)

    return {
        "Top Possible Diagnoses": top_diseases,
        "Suggested Next Symptoms to Ask": list(symptoms_to_ask)[:10]
    }

# --- Test the Logic ---
# Note: PrimeKG uses formal medical terms, so we use 'headache' and 'nausea'
extracted_patient_symptoms = ['headache', 'nausea']
results = get_next_symptom_candidates(G, extracted_patient_symptoms)

print("--- Graph RAG Traversal Results ---")
print(f"Patient reports: {extracted_patient_symptoms}\n")
print(f"Top Suspected Diseases:\n{results['Top Possible Diagnoses']}\n")
print(f"Agent should ask about these symptoms next:\n{results['Suggested Next Symptoms to Ask']}")

--- Graph RAG Traversal Results ---
Patient reports: ['headache', 'nausea']

Top Suspected Diseases:
['episodic ataxia', 'hyperthermia, cutaneous, with headaches and nausea', 'monosodium glutamate sensitivity']

Agent should ask about these symptoms next:
['muscle weakness', 'paresthesia', 'autosomal recessive inheritance', 'juvenile onset', 'spasticity', 'vestibular dysfunction', 'reduced visual acuity', 'arachnoid cyst', 'seizure', 'calf muscle hypertrophy']


In [7]:
!pip install -q spacy negspacy

In [8]:
import spacy
from negspacy.negation import Negex

## Medical NER

### HF: BioMed_NER

In [9]:
from transformers import pipeline

# Load the DeBERTa-based biomedical NER model
print("Loading Hugging Face NER pipeline...")
hf_ner = pipeline(
    "token-classification",
    model="Helios9/BioMed_NER",
    aggregation_strategy="simple" # Groups sub-words into full entities
)

def extract_with_hf(text):
    # Run the pipeline
    entities = hf_ner(text)

    # Filter for symptoms/diseases
    # (Note: exact label strings depend on the specific HF model's training data)
    symptoms = []
    for ent in entities:
        if ent['entity_group'] in ['Disease_disorder', 'Sign_symptom']:
            symptoms.append(ent['word'].lower())

    return list(set(symptoms))



Loading Hugging Face NER pipeline...


config.json:   0%|          | 0.00/5.14k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  736MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

...

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/285 [00:00<?, ?B/s]

In [10]:
# Test it
test_text = "I've had a terrible headache and some nausea since yesterday, but I don't have a fever, and I deny any chest pain."
extracted = extract_with_hf(test_text)

print(f"Extracted Symptoms: {extracted}")

Extracted Symptoms: ['chest pain', 'headache', 'nausea', 'fever']


### LLM

In [11]:
! pip install -q gliner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.6/245.6 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 84.5 MB/s eta 0:00:00


In [12]:
from gliner import GLiNER
import time

In [13]:
gliner_model = GLiNER.from_pretrained("gliner-community/gliner_medium-v2.5")
labels = ["Symptom", "Disease"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

In [14]:
print("Loading spaCy base...")
nlp = spacy.load("en_core_web_sm")

# Add NegEx to the pipeline, telling it to evaluate GLiNER's "Symptom" label
nlp.add_pipe("negex", config={"ent_types": ["Symptom", "Disease"]})

Loading spaCy base...


In [15]:
# --- Test Data ---
test_sentences = [
    "I've had a terrible headache and some nausea since yesterday, but I don't have a fever.",
    "The patient denies experiencing any chest pain or shortness of breath, but reports mild dizziness.",
    "History of type 2 diabetes, currently presenting with acute lower back pain."
]

results = []

print("Running inference...")
for text in test_sentences:

    # 1. Evaluate scispaCy
    t0 = time.time()
    doc = nlp(text)
    # Check the ._.negex flag attached by the pipeline
    spacy_ents = [f"{ent.text} (Negated)" if ent._.negex else f"{ent.text} (Positive)" for ent in doc.ents if ent.label_ == "DISEASE"]
    t_spacy = time.time() - t0

    # 2. Evaluate HF Transformer
    t0 = time.time()
    hf_out = hf_ner(text)
    hf_ents = [f"{ent['word']} ({ent['entity_group']})" for ent in hf_out]
    t_hf = time.time() - t0

    # 3. Evaluate GLiNER
    t0 = time.time()
    gliner_out = gliner_model.predict_entities(text, labels, threshold=0.4)
    gliner_ents = [f"{ent['text']} ({ent['label']})" for ent in gliner_out]
    t_gliner = time.time() - t0

    # Store results
    results.append({
        "Patient Input": text,
        "scispaCy\n(Rules)": "\n".join(spacy_ents) if spacy_ents else "None",
        "scispaCy Time": f"{t_spacy:.3f}s",
        "Transformer\n(Pre-trained)": "\n".join(hf_ents) if hf_ents else "None",
        "HF Time": f"{t_hf:.3f}s",
        "GLiNER\n(Zero-shot)": "\n".join(gliner_ents) if gliner_ents else "None",
        "GLiNER Time": f"{t_gliner:.3f}s"
    })



Running inference...


In [16]:
df = pd.DataFrame(results)
display(df)

,Patient Input,scispaCy\n(Rules),scispaCy Time,Transformer\n(Pre-trained),HF Time,GLiNER\n(Zero-shot),GLiNER Time
0,I've had a terrible headache and some nausea s...,None,0.116s,terrible (Severity)\nheadache (Sign_symptom)\n...,1.066s,terrible headache (Symptom)\nnausea (Symptom)\...,1.573s
1,The patient denies experiencing any chest pain...,None,0.042s,chest pain (Sign_symptom)\nshortness of breath...,0.114s,chest pain (Symptom)\nshortness of breath (Sym...,1.225s
2,"History of type 2 diabetes, currently presenti...",None,0.064s,type 2 diabetes (History)\nacute (Detailed_des...,0.273s,type 2 diabetes (Disease)\nacute lower back pa...,0.593s


In [17]:
def extract_hybrid_state(text):
    # Step A: Let GLiNER find the complex entities
    gliner_ents = gliner_model.predict_entities(text, labels, threshold=0.4)

    # Step B: Prepare the spaCy document
    doc = nlp(text)

    # Step C: Map GLiNER's character offsets to spaCy's token spans
    spacy_spans = []
    for ent in gliner_ents:
        # doc.char_span accurately aligns the raw text offsets to tokens
        span = doc.char_span(ent['start'], ent['end'], label=ent['label'])
        if span is not None:
            spacy_spans.append(span)

    # Inject our GLiNER entities into the spaCy document
    doc.ents = spacy_spans

    # Step D: Run the document through the NegEx pipeline component
    doc = nlp.get_pipe("negex")(doc)

    # Step E: Format the output for our Graph RAG
    state = {"positive_symptoms": [], "negated_symptoms": []}

    for ent in doc.ents:
        if ent._.negex:
            state["negated_symptoms"].append(ent.text.lower())
        else:
            state["positive_symptoms"].append(ent.text.lower())

    return state

# --- Test the Hybrid Approach ---
test_text = "I've had a terrible headache and some nausea since yesterday, but I don't have a fever."
hybrid_state = extract_hybrid_state(test_text)

print("--- Hybrid Extraction Results ---")
print(f"Text: {test_text}")
print(f"Positive (To query): {hybrid_state['positive_symptoms']}")
print(f"Negated (To ignore): {hybrid_state['negated_symptoms']}")

--- Hybrid Extraction Results ---
Text: I've had a terrible headache and some nausea since yesterday, but I don't have a fever.
Positive (To query): ['terrible headache', 'nausea']
Negated (To ignore): ['fever']


## Generation

In [18]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.7 MB/s eta 0:00:00


In [19]:
import os
from groq import Groq
from google.colab import userdata

# Retrieve the API key securely from Colab Secrets
groq_api_key = userdata.get('GROQ_API_KEY')

# Initialize the Groq client
client = Groq(api_key=groq_api_key)

print("Groq Client successfully initialized!")

Groq Client successfully initialized!


In [20]:
# The System Prompt instructs the LLM on its exact persona and operational boundaries.
SYSTEM_PROMPT = """You are an expert clinical triage AI. Your job is to interview patients and gather their History of Present Illness (HPI).
You will be provided with the patient's current state and a list of 'Candidate Symptoms' generated by a Medical Knowledge Graph.

YOUR DIRECTIVES:
1. Empathy first: Acknowledge the patient's distress briefly before asking a question.
2. One question at a time: Ask exactly ONE question to evaluate ONE of the Candidate Symptoms. Do NOT overwhelm the patient.
3. Constraint: NEVER ask about a symptom listed in 'Denied Symptoms'.
4. Stopping Criteria: Evaluate the patient's positive symptoms against the SOCRATES criteria (Site, Onset, Character, Radiation, Associated symptoms, Time course, Exacerbating factors, Severity).
   - If you have asked 4 questions OR feel you have enough information, output exactly the phrase: "[STOP_INTERVIEW]" followed by a brief summary of the suspected clinical picture.

Do not play doctor. You are gathering information, not prescribing."""

def generate_next_question_groq(patient_utterance, extracted_state, graph_candidates):
    """
    Combines the NLP extraction and Graph traversal into a prompt for gpt-oss-120b via Groq.
    """

    # 1. Format the dynamic context for the LLM
    context_block = f"""
[CLINICAL CONTEXT]
Patient's Latest Message: "{patient_utterance}"
Confirmed Positive Symptoms: {extracted_state['positive_symptoms']}
Denied Symptoms (Do NOT ask about these): {extracted_state['negated_symptoms']}
Knowledge Graph Suggested Symptoms to Explore: {graph_candidates}
    """

    # 2. Call the Groq API
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": context_block + "\nBased on the clinical context, what is the single next best question to ask the patient?"}
        ],
        temperature=0.3, # Low temperature for clinical reasoning stability
        # max_tokens=1000
    )

    return response.choices[0].message.content.strip()

In [21]:
import json
import re

# --- 1. Updated Agent System Prompt (Forcing JSON with Options) ---
STRUCTURED_SYSTEM_PROMPT = """You are an expert clinical triage AI interviewing a patient.
You will be given:
- Current positive symptoms
- Denied symptoms
- Knowledge Graph suggested symptoms to evaluate

YOUR GOAL:
1. Choose ONE primary target symptom from the suggested symptoms to evaluate next.
2. Formulate a polite, empathetic question about it.
3. Provide EXACTLY 3 numbered options for the patient to choose from:
   - Option 1: Positive / Severe statement confirming the symptom.
   - Option 2: Moderate / Partial statement about the symptom.
   - Option 3: Negative statement denying the symptom.
4. Evaluate if you have enough information to stop (SOCRATES criteria / 4+ questions asked).
5. Track the SOCRATES criteria for the patient's primary complaint.

OUTPUT FORMAT:
You MUST reply in pure, valid JSON with this exact structure:
{
  "target_symptom": "name of symptom being evaluated",
  "empathy_and_question": "Brief empathetic remark + single question",
  "options": [
    "Option 1 description",
    "Option 2 description",
    "Option 3 description"
  ],
  "socrates_tracker": {
    "site": true/false,
    "onset": true/false,
    "character": true/false,
    "radiation": true/false,
    "associated_symptoms": true/false,
    "time_course": true/false,
    "exacerbating_relieving": true/false,
    "severity": true/false
  },
  "socrates_score": "integer (count of true values, 0-8)",
  "is_complete": false,
  "summary_if_complete": ""
}

CRITICAL: Output ONLY the raw JSON object. Do NOT wrap the JSON in markdown formatting (```json). Do NOT add conversational text before or after the JSON.
"""

def generate_structured_question_groq(patient_utterance, extracted_state, graph_candidates, retries=2):
    """
    Calls gpt-oss-120b on Groq and returns a structured JSON payload.
    Includes a retry mechanism in case the JSON validation fails.
    """
    context_block = f"""
[CLINICAL CONTEXT]
Patient's Latest Message: "{patient_utterance}"
Confirmed Positive Symptoms: {extracted_state['positive_symptoms']}
Denied Symptoms (Do NOT ask about these): {extracted_state['negated_symptoms']}
Knowledge Graph Suggested Symptoms to Explore: {graph_candidates}
    """

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model="openai/gpt-oss-120b",
                messages=[
                    {"role": "system", "content": STRUCTURED_SYSTEM_PROMPT},
                    {"role": "user", "content": context_block + "\nGenerate the next structured question. Output ONLY raw JSON."}
                ],
                response_format={"type": "json_object"},
                temperature=0.2,
                # max_tokens=800 # Increased from 300 to prevent JSON truncation
            )

            return json.loads(response.choices[0].message.content)

        except Exception as e:
            if attempt == retries - 1:
                print(f"[System Error] JSON generation failed after {retries} attempts. Error: {e}")
                # Fallback safe payload to prevent crash
                return {
                    "target_symptom": None,
                    "empathy_and_question": "I apologize, I experienced a technical glitch. Could you please tell me more about how you are feeling?",
                    "options": ["1. I feel worse", "2. I feel about the same", "3. I feel better"],
                    "is_complete": False
                }
            print(f"[System Warning] Retrying JSON generation (Attempt {attempt+2}/{retries})...")


# --- 2. Context-Aware Master Chat Loop ---
def run_interactive_triage(max_turns=10):
    print("======================================================")
    print("🩺 Agentic Graph RAG Triage System (Options & Context-Aware)")
    print("Type 'exit' or 'quit' to stop.")
    print("======================================================\n")

    patient_state = {
        "positive_symptoms": set(),
        "negated_symptoms": set()
    }

    last_target_symptom = None
    last_options = []

    print("Assistant: Hello! I'm your clinical triage assistant. What symptoms are you experiencing today?")

    turn_count = 0
    while turn_count < max_turns:
        raw_user_input = input("\nPatient: ").strip()

        if raw_user_input.lower() in ['exit', 'quit', 'stop']:
            print("Ending interview.")
            break

        # --- Step A: Resolve Numeric Selection (1, 2, 3) or Yes/No Context ---
        resolved_input = raw_user_input

        # 1. If user typed a number (1, 2, or 3), map it to the corresponding text option
        if raw_user_input in ['1', '2', '3'] and len(last_options) == 3:
            resolved_input = last_options[int(raw_user_input) - 1]
            print(f"  [System interpreted selection: '{resolved_input}']")

        # 2. Extract entities from resolved input using Hybrid NER
        extracted = extract_hybrid_state(resolved_input)

        # 3. Fallback Context Resolver for direct "Yes" / "No" without extracted entities
        lower_input = resolved_input.lower()
        is_yes = any(w in lower_input for w in ['yes', 'yeah', 'yep', 'sure', 'i do', 'positive', 'true'])
        is_no = any(w in lower_input for w in ['no', 'nope', 'nah', 'don\'t', 'denied', 'false', 'none'])

        if last_target_symptom:
            if is_yes and last_target_symptom not in extracted["positive_symptoms"]:
                extracted["positive_symptoms"].append(last_target_symptom)
            elif is_no and last_target_symptom not in extracted["negated_symptoms"]:
                extracted["negated_symptoms"].append(last_target_symptom)

        # --- Step B: Update Patient State ---
        patient_state["positive_symptoms"].update(extracted["positive_symptoms"])
        patient_state["negated_symptoms"].update(extracted["negated_symptoms"])
        patient_state["positive_symptoms"] -= patient_state["negated_symptoms"]

        current_positives = list(patient_state["positive_symptoms"])
        current_negatives = list(patient_state["negated_symptoms"])

        # --- Step C: Query Graph RAG ---
        graph_results = get_next_symptom_candidates(G, current_positives)
        candidates = []
        if isinstance(graph_results, dict):
            candidates = graph_results.get("Suggested Next Symptoms to Ask", [])

        # --- Step D: Generate Structured Agent Question ---
        llm_state = {
            "positive_symptoms": current_positives,
            "negated_symptoms": current_negatives
        }

        response_json = generate_structured_question_groq(resolved_input, llm_state, candidates)

        # --- Step E: Display Assistant Response ---
        print(f"\nAssistant: {response_json['empathy_and_question']}\n")

        # Check for completion
        if response_json.get("is_complete"):
            print("======================================================")
            print("📝 Triage Complete. Summary for Doctor:")
            print(f"- Confirmed Symptoms: {current_positives}")
            print(f"- Denied Symptoms: {current_negatives}")
            if isinstance(graph_results, dict):
                print(f"- Top Suspected Diagnoses: {graph_results.get('Top Possible Diagnoses', [])}")
            print(f"- Summary: {response_json.get('summary_if_complete')}")
            print("======================================================")
            break

        # Store state for next turn resolution
        last_target_symptom = response_json.get("target_symptom")
        last_options = response_json.get("options", [])

        # Render the 3 options for the patient
        print("Please choose an option (type 1, 2, or 3) or write your own answer:")
        for idx, option in enumerate(last_options, 1):
            print(f"  {idx}. {option}")

        turn_count += 1

# --- Run the Updated Pipeline ---
# run_interactive_triage()

In [22]:
def translate_patient_to_english(arabic_text):
    """
    INBOUND LAYER: Translates the patient's Egyptian Arabic input into English.
    Designed to understand colloquial medical expressions.
    """
    prompt = f"""Translate the following Egyptian Arabic medical complaint into clear, clinical English.
    Translate slang appropriately (e.g., 'مغص' -> abdominal cramps/colic).
    Output ONLY the English translation, nothing else.

    Patient: "{arabic_text}"
    Translation:"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
        # max_tokens=150
    )
    return response.choices[0].message.content.strip()

def translate_agent_to_arabic(english_json_response):
    """
    OUTBOUND LAYER: Translates the Agent's English JSON decision back into Egyptian Arabic.
    Ensures the patient feels they are speaking to a local doctor.
    """
    prompt = f"""Translate the following English clinical JSON into Egyptian Arabic (Masri).
    Use polite, empathetic Egyptian dialect for the question (e.g., 'ألف سلامة عليك', 'حاسس بـ...').
    Maintain the exact JSON structure. Do NOT wrap in markdown.

    English JSON:
    {json.dumps(english_json_response, indent=2)}
    """

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
        temperature=0.2,
        # max_tokens=600
    )
    return json.loads(response.choices[0].message.content)

In [23]:
def run_arabic_interactive_triage(max_turns=10):
    print("======================================================")
    print("🩺 Agentic Graph RAG Triage System (Egyptian Arabic)")
    print("Type 'خروج' or 'exit' to stop.")
    print("======================================================\n")

    patient_state = {
        "positive_symptoms": set(),
        "negated_symptoms": set()
    }

    last_target_symptom = None
    last_options_arabic = []
    last_options_english = []

    print("المساعد: أهلاً بيك. ألف سلامة عليك، تقدر تقولي حاسس بإيه أو بتشتكي من إيه النهاردة؟")

    turn_count = 0
    while turn_count < max_turns:
        raw_user_input = input("\nالمريض (Patient): ").strip()

        if raw_user_input.lower() in ['خروج', 'exit', 'quit', 'stop']:
            break

        # --- 1. RESOLVE SELECTIONS ---
        resolved_arabic = raw_user_input
        resolved_english = ""

        # If user typed a number (1, 2, or 3)
        if raw_user_input in ['1', '2', '3', '١', '٢', '٣'] and len(last_options_arabic) == 3:
            # Standardize Arabic numerals to English integers for indexing
            idx = int(raw_user_input.translate(str.maketrans('١٢٣', '123'))) - 1
            resolved_arabic = last_options_arabic[idx]
            # Map it directly to the English equivalent for the internal system!
            resolved_english = last_options_english[idx]
            print(f"  [System matched to: '{resolved_arabic}']")
        else:
            # --- 2. INBOUND TRANSLATION (Arabic -> English) ---
            resolved_english = translate_patient_to_english(resolved_arabic)
            print(f"  [Internal Translation: '{resolved_english}']")

        # --- 3. INTERNAL ENGLISH PIPELINE ---
        # Extract entities using Hybrid NER on the ENGLISH translation
        extracted = extract_hybrid_state(resolved_english)

        # Handle Yes/No context in English
        lower_en = resolved_english.lower()
        is_yes = any(w in lower_en for w in ['yes', 'yeah', 'yep', 'positive', 'true', 'do have'])
        is_no = any(w in lower_en for w in ['no', 'nope', 'denied', 'false', 'don\'t', 'do not'])

        if last_target_symptom:
            if is_yes and last_target_symptom not in extracted["positive_symptoms"]:
                extracted["positive_symptoms"].append(last_target_symptom)
            elif is_no and last_target_symptom not in extracted["negated_symptoms"]:
                extracted["negated_symptoms"].append(last_target_symptom)

        # Update State
        patient_state["positive_symptoms"].update(extracted["positive_symptoms"])
        patient_state["negated_symptoms"].update(extracted["negated_symptoms"])
        patient_state["positive_symptoms"] -= patient_state["negated_symptoms"]

        current_positives = list(patient_state["positive_symptoms"])
        current_negatives = list(patient_state["negated_symptoms"])

        # Graph RAG
        graph_results = get_next_symptom_candidates(G, current_positives)
        candidates = graph_results.get("Suggested Next Symptoms to Ask", []) if isinstance(graph_results, dict) else []

        # Agent Decision
        llm_state = {"positive_symptoms": current_positives, "negated_symptoms": current_negatives}
        english_json = generate_structured_question_groq(resolved_english, llm_state, candidates)

        # --- 4. OUTBOUND TRANSLATION (English -> Arabic) ---
        if english_json.get("is_complete"):
            print("\n======================================================")
            print("⚙️ Triage Complete. Generating Internal English Report...")
            final_graph_evidence = calculate_diagnostic_evidence(G, patient_state, top_k=3)
            final_report = generate_diagnostic_report_groq(llm_state, final_graph_evidence)
            print("======================================================")
            print("📋 FINAL CLINICAL REPORT (For Doctor)")
            for idx, ddx in enumerate(final_report["top_diagnoses"], 1):
                print(f"[{idx}] Diagnosis: {ddx['diagnosis'].upper()} (Probability: {ddx['estimated_probability']})")
                print(f"    Evidence:  {ddx['graph_evidence']}\n")
            break

        arabic_json = translate_agent_to_arabic(english_json)

        # --- 5. RENDER UI ---
        print(f"\nالمساعد: {arabic_json['empathy_and_question']}\n")
        print("اختار رقم من الاختيارات دي (1، 2، 3) أو اكتب ردك:")

        last_options_arabic = arabic_json.get("options", [])
        last_options_english = english_json.get("options", []) # Save English for mapping next turn
        last_target_symptom = english_json.get("target_symptom")

        for idx, option in enumerate(last_options_arabic, 1):
            print(f"  {idx}. {option}")

        turn_count += 1

# Start the Arabic Sandwich Loop!
run_arabic_interactive_triage()

🩺 Agentic Graph RAG Triage System (Egyptian Arabic)
Type 'خروج' or 'exit' to stop.

المساعد: أهلاً بيك. ألف سلامة عليك، تقدر تقولي حاسس بإيه أو بتشتكي من إيه النهاردة؟

المريض (Patient): عندي دور برد
  [Internal Translation: 'I have a cold.']

المساعد: ألف سلامة عليك، أنا فاهم إن الزكام ممكن يكون مزعج. هل بتحس بارتفاع في الحرارة؟

اختار رقم من الاختيارات دي (1، 2، 3) أو اكتب ردك:
  1. أيوة، عندي حرارة عالية (أعلى من 102°F)
  2. عندي حرارة خفيفة أو متوسطة (أقل من 102°F)
  3. لا، ما عنديش حرارة

المريض (Patient): 3
  [System matched to: 'لا، ما عنديش حرارة']

المساعد: أنا عارف إنك بتعاني من نزلة برد. هل عندك سعال؟

اختار رقم من الاختيارات دي (1، 2، 3) أو اكتب ردك:
  1. عندي سعال متكرر وشديد بيخليني ما أنامش
  2. عندي سعال خفيف ومش دايم
  3. ما عنديش سعال

المريض (Patient): 2
  [System matched to: 'عندي سعال خفيف ومش دايم']

المساعد: أنا فاهم إن السعال ممكن يكون مزعج. ممكن تحكيلي أكتر عنه؟

اختار رقم من الاختيارات دي (1، 2، 3) أو اكتب ردك:
  1. عندي سعال متكرر وشديد بيوجعني جدًا.
  2. عند

In [24]:
def evaluate_stopping_criteria(
    turn_count,
    patient_state,
    top_diagnoses_metrics,
    socrates_coverage,
    red_flags_checked
):
    """
    Evaluates whether the clinical interview should stop or continue.
    Returns: (should_stop: bool, reason: str)
    """
    # 1. HARD BOUNDS
    if turn_count < 3:
        return False, "Minimum turns (3) not reached yet."

    if turn_count >= 7:
        return True, "Maximum turns reached (patient fatigue limit)."

    # 2. SAFETY RULE: Are all red flag symptoms for top suspected diseases checked?
    if not red_flags_checked:
        return False, "Critical red flag symptoms still need to be ruled out."

    # 3. STATISTICAL CONFIDENCE & ENTROPY
    probabilities = [d['probability'] for d in top_diagnoses_metrics]

    if len(probabilities) > 1:
        # Calculate Margin between top 1 and top 2
        margin = probabilities[0] - probabilities[1]

        # High confidence in top diagnosis
        if probabilities[0] >= 0.75 or margin >= 0.45:
            return True, f"High diagnostic certainty reached (Top match: {probabilities[0]*100:.1f}%)."

    # 4. SOCRATES COMPLETENESS CHECK
    socrates_score = sum(socrates_coverage.values()) # e.g., 5 out of 8 slots filled

    if socrates_score >= 5 and margin >= 0.30:
        return True, "Sufficient SOCRATES history collected with moderate-to-high confidence."

    return False, "Further inquiry needed to differentiate top diagnoses."

In [25]:
import math

def calculate_statistical_confidence(disease_metrics):
    """
    Converts raw graph scores into probabilities and calculates entropy.
    """
    scores = [data['raw_confidence_score'] for data in disease_metrics.values()]

    if not scores or sum(scores) == 0:
        return {"entropy": 1.0, "margin": 0.0, "probabilities": []}

    # Normalize scores into a probability distribution (summing to 1.0)
    total_score = sum(scores)
    probabilities = [s / total_score for s in scores]

    # Calculate Shannon Entropy
    # Lower entropy means the system is highly focused on 1 or 2 diagnoses.
    entropy = -sum(p * math.log2(p) for p in probabilities if p > 0)

    # Calculate Probability Margin (Difference between Top 1 and Top 2)
    margin = (probabilities[0] - probabilities[1]) if len(probabilities) > 1 else probabilities[0]

    return {
        "entropy": round(entropy, 3),
        "margin": round(margin, 3),
        "probabilities": [round(p, 3) for p in probabilities]
    }

In [26]:
def should_stop_interview(turn_count, max_turns, stats, llm_json_output, red_flags_cleared=True):
    """
    Evaluates the 4 pillars to decide if the interview is complete.
    """
    print("\n--- 🧠 Diagnostic Engine Metrics ---")
    print(f"Turns: {turn_count}/{max_turns} | Entropy: {stats['entropy']} | Margin: {stats['margin']}")
    print(f"SOCRATES Score: {llm_json_output.get('socrates_score', 0)}/8")

    # 1. Hard Turn Limits
    if turn_count < 3:
        return False, "Minimum turns not reached."
    if turn_count >= max_turns:
        return True, "Maximum turns reached (patient fatigue safety limit)."

    # 2. Safety (Mocked for POC)
    if not red_flags_cleared:
        return False, "Red flags pending."

    # 3. Diagnostic Convergence (High Margin & Low Entropy)
    if stats['margin'] >= 0.45 and stats['entropy'] < 1.2:
        return True, "High diagnostic certainty reached based on graph data."

    # 4. Clinical Completeness (SOCRATES)
    if llm_json_output.get('socrates_score', 0) >= 5 and stats['margin'] >= 0.25:
        return True, "Sufficient clinical history gathered with moderate certainty."

    return False, "Continuing to gather evidence."

In [27]:
def calculate_diagnostic_evidence(graph, patient_state, top_k=3):
    """
    Calculates a heuristic confidence score for diseases based on PrimeKG overlaps.
    Penalizes diseases if the patient explicitly denied associated symptoms.
    """
    positive = set(patient_state["positive_symptoms"])
    negated = set(patient_state["negated_symptoms"])

    disease_metrics = {}

    # Step A: Find all diseases connected to the positive symptoms
    for symptom in positive:
        if symptom in graph:
            for neighbor in graph.neighbors(symptom):
                if graph.nodes[neighbor].get('node_type') == 'disease':
                    if neighbor not in disease_metrics:

                        # Step B: Retrieve the complete symptom profile for this disease
                        disease_profile = set(
                            n for n in graph.neighbors(neighbor)
                            if graph.nodes[n].get('node_type') == 'effect/phenotype'
                        )

                        matched = positive.intersection(disease_profile)
                        conflicting = negated.intersection(disease_profile)

                        # Step C: Calculate the raw score
                        # (Matches minus heavy penalty for conflicting symptoms)
                        raw_score = len(matched) - (len(conflicting) * 1.5)

                        # Step D: Normalize the score
                        confidence = raw_score / max(len(disease_profile), 1)

                        disease_metrics[neighbor] = {
                            "raw_confidence_score": max(round(confidence, 3), 0.001),
                            "matched_evidence": list(matched),
                            "conflicting_evidence": list(conflicting)
                        }

    # Fallback if no matching disease nodes were hit in graph
    if not disease_metrics:
        return {"unspecified symptom cluster": {"raw_confidence_score": 0.001, "matched_evidence": [], "conflicting_evidence": []}}

    # Step E: Sort by highest confidence and extract Top K
    sorted_diseases = sorted(
        disease_metrics.items(),
        key=lambda x: x[1]['raw_confidence_score'],
        reverse=True
    )

    top_diagnoses = {}
    for disease, metrics in sorted_diseases[:top_k]:
        top_diagnoses[disease] = metrics

    return top_diagnoses


# --- Final Report Generator ---
DIAGNOSTIC_REPORT_PROMPT = """You are an expert Chief Medical Officer. The triage agent has finished gathering the patient's history.
You are provided with:
1. The Patient's Clinical State (Positives and Negated symptoms).
2. The Top Diagnostic Matches retrieved directly from the PrimeKG Knowledge Graph, including mathematical confidence scores.

YOUR GOAL:
Generate a formal clinical diagnostic report evaluating the top probable options (differential diagnoses).
For each diagnosis, you must translate the raw confidence score into an estimated probability percentage (e.g., 85%).
You must explicitly state the reasoning using the graph evidence.

OUTPUT FORMAT:
You MUST reply in pure, valid JSON with this exact structure:
{
  "top_diagnoses": [
    {
      "diagnosis": "Name of disease",
      "estimated_probability": "XX%",
      "reasoning": "Clinical justification based on the matched vs. conflicting symptoms.",
      "graph_evidence": "(Patient) -> [has_symptom] -> (Symptom X) <- [associated_with] <- (Disease Y)"
    }
  ],
  "triage_recommendation": "Brief clinical recommendation for the attending human doctor."
}
"""

def generate_diagnostic_report_groq(patient_state, graph_evidence):
    """
    Calls the Groq API to format the mathematical graph evidence into a final JSON report for the doctor.
    """
    context_block = f"""
[FINAL CLINICAL STATE]
Positive Symptoms: {patient_state['positive_symptoms']}
Denied Symptoms: {patient_state['negated_symptoms']}

[KNOWLEDGE GRAPH EVIDENCE]
{json.dumps(graph_evidence, indent=2)}
    """

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "system", "content": DIAGNOSTIC_REPORT_PROMPT},
            {"role": "user", "content": context_block + "\nGenerate the final diagnostic report."}
        ],
        response_format={"type": "json_object"},
        temperature=0.1,
        max_tokens=800
    )

    return json.loads(response.choices[0].message.content)

In [28]:
def run_arabic_agentic_triage(max_turns=10):
    print("======================================================")
    print("🩺 Agentic Graph RAG Triage System (Egyptian Arabic & Stopping Logic)")
    print("Type 'خروج' or 'exit' to stop.")
    print("======================================================\n")

    patient_state = {"positive_symptoms": set(), "negated_symptoms": set()}
    last_target_symptom = None
    last_options_arabic = []
    last_options_english = []

    print("المساعد: أهلاً بيك. ألف سلامة عليك، تقدر تقولي حاسس بإيه أو بتشتكي من إيه النهاردة؟")

    turn_count = 0
    while turn_count < max_turns:
        raw_user_input = input("\nالمريض (Patient): ").strip()
        if raw_user_input.lower() in ['خروج', 'exit', 'quit', 'stop']:
            break

        # --- A. Input Resolution & Inbound Translation ---
        resolved_english = ""
        if raw_user_input in ['1', '2', '3', '١', '٢', '٣'] and len(last_options_arabic) == 3:
            idx = int(raw_user_input.translate(str.maketrans('١٢٣', '123'))) - 1
            resolved_english = last_options_english[idx]
        else:
            resolved_english = translate_patient_to_english(raw_user_input)

        # --- B. NER Extraction & State Tracking ---
        extracted = extract_hybrid_state(resolved_english)

        # Yes/No Context mapping
        lower_en = resolved_english.lower()
        if last_target_symptom:
            if any(w in lower_en for w in ['yes', 'yeah', 'yep', 'positive']):
                extracted["positive_symptoms"].append(last_target_symptom)
            elif any(w in lower_en for w in ['no', 'nope', 'denied', 'false']):
                extracted["negated_symptoms"].append(last_target_symptom)

        patient_state["positive_symptoms"].update(extracted["positive_symptoms"])
        patient_state["negated_symptoms"].update(extracted["negated_symptoms"])
        patient_state["positive_symptoms"] -= patient_state["negated_symptoms"]

        current_positives = list(patient_state["positive_symptoms"])
        current_negatives = list(patient_state["negated_symptoms"])

        # --- C. Graph Diagnostics & Statistical Math ---
        # 1. Get exact graph evidence
        graph_evidence = calculate_diagnostic_evidence(G, patient_state, top_k=3)
        # 2. Convert evidence into entropy & margin stats
        stats = calculate_statistical_confidence(graph_evidence)
        # 3. Get next suggested symptoms
        graph_results = get_next_symptom_candidates(G, current_positives)
        candidates = graph_results.get("Suggested Next Symptoms to Ask", []) if isinstance(graph_results, dict) else []

        # --- D. LLM Agent Call (Generates Next Question & Tracks SOCRATES) ---
        llm_state = {"positive_symptoms": current_positives, "negated_symptoms": current_negatives}
        english_json = generate_structured_question_groq(resolved_english, llm_state, candidates)

        turn_count += 1

        # --- E. Evaluate Stopping Criteria ---
        stop_flag, stop_reason = should_stop_interview(turn_count, max_turns, stats, english_json)

        if stop_flag or english_json.get("is_complete"):
            print(f"\n[System Halt] Reason: {stop_reason}")
            print("======================================================")
            print("⚙️ Triage Complete. Generating Internal English Report...")

            final_report = generate_diagnostic_report_groq(llm_state, graph_evidence)

            print("======================================================")
            print("📋 FINAL CLINICAL REPORT (For Doctor Dashboard)")
            for idx, ddx in enumerate(final_report["top_diagnoses"], 1):
                print(f"[{idx}] Diagnosis: {ddx['diagnosis'].upper()} (Probability: {ddx['estimated_probability']})")
                print(f"    Evidence:  {ddx['graph_evidence']}\n")
            print("======================================================")

            # Optional: Translate a brief goodbye to the patient
            print("المساعد: شكراً جداً لتعاونك. المعلومات دي هتتبعت للدكتور حالاً وهيدخل يتابع معاك.")
            break

        # --- F. Outbound Translation & UI Render ---
        arabic_json = translate_agent_to_arabic(english_json)

        print(f"\nالمساعد: {arabic_json['empathy_and_question']}\n")
        print("اختار رقم من الاختيارات دي (1، 2، 3) أو اكتب ردك:")

        last_options_arabic = arabic_json.get("options", [])
        last_options_english = english_json.get("options", [])
        last_target_symptom = english_json.get("target_symptom")

        for idx, option in enumerate(last_options_arabic, 1):
            print(f"  {idx}. {option}")



In [29]:
# Run the complete system
run_arabic_agentic_triage()

🩺 Agentic Graph RAG Triage System (Egyptian Arabic & Stopping Logic)
Type 'خروج' or 'exit' to stop.

المساعد: أهلاً بيك. ألف سلامة عليك، تقدر تقولي حاسس بإيه أو بتشتكي من إيه النهاردة؟

المريض (Patient): دور برد

--- 🧠 Diagnostic Engine Metrics ---
Turns: 1/10 | Entropy: -0.0 | Margin: 1.0
SOCRATES Score: 0/8

المساعد: ألف سلامة عليك، أنا فاهم إنك بتعاني من نزلة برد شائعة. هل عندك سعال؟

اختار رقم من الاختيارات دي (1، 2، 3) أو اكتب ردك:
  1. أيوة، عندي سعال شديد ومُستمر.
  2. عندي سعال خفيف ومُتناوب.
  3. لا، ما عنديش سعال.

المريض (Patient): 2

--- 🧠 Diagnostic Engine Metrics ---
Turns: 2/10 | Entropy: -0.0 | Margin: 1.0
SOCRATES Score: 1/8

المساعد: ألف سلامة عليك، أنا فاهم إن السعال ممكن يكون مزعج. ممكن تقولي إمتى لاحظت السعال لأول مرة؟

اختار رقم من الاختيارات دي (1، 2، 3) أو اكتب ردك:
  1. أيوة، عندى سعال متكرر وشديد بيصحىنى بالليل.
  2. عندي سعال بييجي ويمشي، ساعات بيكون مزعج.
  3. لأ، ما عنديش سعال.

المريض (Patient): بدا من امبارح

--- 🧠 Diagnostic Engine Metrics ---
Turns: 3/1